# 1. Environment Setup & Data Ingestion

In [58]:
# Install an older version (2.18.0) to bypass script blocking
!pip install "datasets==2.18.0"
from datasets import load_dataset

In [59]:
# Install required libraries: pip install datasets spacy
import re
import pandas as pd
import spacy

# 2. Document Parsing & Structuring

In [60]:
# 1. Load dataset (HuggingFace midas/krapivin)
dataset = load_dataset("midas/krapivin", "raw")

def segment_krapivin_doc(tokens):
    sections = {}
    current_section = None
    content = []

    i = 0
    while i < len(tokens):
        # Start new section when '--' is encountered
        if tokens[i] == '--' and i + 1 < len(tokens):
            # Save previous section
            if current_section:
                sections[current_section] = content

            # Set new section type (T, A, etc.)
            current_section = tokens[i+1]
            content = []
            i += 2 # Skip '--' and 'section identifier'
        else:
            content.append(tokens[i])
            i += 1

    # Save the last section
    if current_section:
        sections[current_section] = content

    return sections

# Execution example
sample = dataset['test'][0]
doc_sections = segment_krapivin_doc(sample['document'])

# Mapping to proposal structure
title_tokens = doc_sections.get('T', [])
abstract_tokens = doc_sections.get('A', [])
# Treat everything after 'A' as body if 'B' tag is missing
body_tokens = doc_sections.get('B', []) if 'B' in doc_sections else []

# Output results
print(f"Title ({len(title_tokens)} tokens): {' '.join(title_tokens)}")
print(f"Abstract ({len(abstract_tokens)} tokens): {' '.join(abstract_tokens[:20])}...")
if not body_tokens:
    print("Warning: If 'B' separator is missing, the body may need to be handled after the abstract.")

Title (5 tokens): Detecting graph-based spatial outliers .
Abstract (116 tokens): of outliers can lead to the discovery of unexpected , interesting , and useful knowledge . Existing methods are designed...


In [61]:
def segment_and_structure(tokens):
    """
    Function to separate T, A, B sections based on '--' delimiter
    """
    sections = {'T': [], 'A': [], 'B': []}
    current_key = None

    i = 0
    while i < len(tokens):
        if tokens[i] == '--' and i + 1 < len(tokens):
            next_token = tokens[i+1]
            if next_token in ['T', 'A', 'B']:
                current_key = next_token
                i += 2
                continue

        if current_key:
            sections[current_key].append(tokens[i])
        i += 1

    return {
        'title': " ".join(sections['T']),
        'abstract': " ".join(sections['A']),
        'body': " ".join(sections['B'])
    }

# Process all data
processed_list = []
for sample in dataset['test']:
    structured_content = segment_and_structure(sample['document'])

    processed_list.append({
        'id': sample['id'],
        'title': structured_content['title'],
        'abstract': structured_content['abstract'],
        'body_preview': structured_content['body'][:200] + "...",
        'ground_truth': ", ".join(sample['extractive_keyphrases'])
    })

df = pd.DataFrame(processed_list)
display(df.head())

,id,title,abstract,body_preview,ground_truth
0,502567,Detecting graph-based spatial outliers .,of outliers can lead to the discovery of unexp...,Introduction Data mining is a process to extra...,"outlier detection, spatial data mining"
1,506154,Task assignment with unknown duration .,We consider a distributed server system and as...,"Introduction In recent years , distributed ser...","load balancing, load sharing, fairness, task a..."
2,504212,Analysis and comparison of two general sparse ...,This paper provides a comprehensive study and ...,Introduction We consider the direct solution o...,parallelism
3,502094,Efficient generation of shared RSA keys .,We describe efficient techniques for a number ...,Introduction We present e-cient protocols for ...,"threshold cryptography, rsa, primality testing"
4,507259,Axioms for real-time logics .,This paper presents a complete axiomatization ...,Introduction Many real-time systems are safety...,"real time, temporal logic, completeness, axiom..."


# 3. Text Normalization (Regex-Based Cleaning)

In [62]:
def clean_academic_text(text):
    if not text:
        return ""

    # 1. Remove NLP special tokens like -lrb-, -rrb-
    text = re.sub(r'-(lrb|rrb|lsb|rsb|lcb|rcb)-', ' ', text, flags=re.IGNORECASE)

    # 2. Remove LaTeX equations
    text = re.sub(r'\$.*?\$', '', text)
    text = re.sub(r'\\\[.*?\\\]', '', text)

    # 3. Remove citations
    text = re.sub(r'\[[\d\s, oak-]*\]', '', text)
    text = re.sub(r'\([A-Z][a-z]+ et al\., \d{4}\)', '', text)

    # 4. Remove emails and URLs
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'http\S+', '', text)

    # 5. Remove section numbers
    text = re.sub(r'\b\d+(\.\d+)*\b', ' ', text)

    # 6. Remove special characters (keep alpha, space, hyphen)
    text = re.sub(r'[^a-zA-Z\s\-]', ' ', text)

    # 7. Clean up white spaces
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.strip('-')

    return text.lower()

In [63]:
test_sample = dataset['test'][0]
structured = segment_and_structure(test_sample['document'])

raw_title = structured['title']
raw_abstract = structured['abstract']
raw_body = structured['body']

clean_title = clean_academic_text(raw_title)
clean_abstract = clean_academic_text(raw_abstract)
clean_body = clean_academic_text(raw_body)

print("TITLE CLEANING TEST")
print(f"PREV: {raw_title}")
print(f"NEXT: {clean_title}")
print("-"*50)

TITLE CLEANING TEST
PREV: Detecting graph-based spatial outliers .
NEXT: detecting graph-based spatial outliers
--------------------------------------------------


In [64]:
# Save results to file for verification
with open("cleaning_test_result.txt", "w") as f:
    f.write("ORIGINAL TITLE\n")
    f.write(raw_title + "\n\n")
    f.write("CLEANED TITLE\n")
    f.write(clean_title + "\n")

print("File 'cleaning_test_result.txt' created successfully.")

File 'cleaning_test_result.txt' created successfully.


# 4. Candidate Phrase Extraction (spaCy Pipeline)

In [65]:
# 1. Load spaCy and set custom stopwords
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
custom_stopwords = {
    "introduction", "abstract", "references", "conclusion", "background",
    "method", "result", "discussion", "figure", "table", "et", "al", "eg", "ie", "th"
}

def extract_candidate_phrases(text, section='B'):
    if not text: return []

    # Prevent spaCy from splitting hyphenated words
    text = text.replace('-', '_')
    doc = nlp(text)

    phrases = []
    current_phrase = []

    allowed_pos = {"NOUN", "PROPN", "ADJ"}
    if section in ['T', 'A']:
        allowed_pos.add("VERB")

    for token in doc:
        lemma = token.lemma_.lower().replace('_', '-')
        is_valid = not (token.is_stop or lemma in custom_stopwords) and len(lemma) > 1

        if is_valid and token.pos_ in allowed_pos:
            if token.pos_ == "VERB" and section in ['T', 'A']:
                if current_phrase:
                    phrases.append(" ".join(current_phrase))
                    current_phrase = []
                phrases.append(lemma)
            else:
                current_phrase.append(lemma)
        else:
            if current_phrase:
                phrases.append(" ".join(current_phrase))
                current_phrase = []

    if current_phrase:
        phrases.append(" ".join(current_phrase))

    return phrases

In [66]:
# 2. Apply spaCy by section
final_title_tokens = extract_candidate_phrases(clean_title, 'T')
final_abstract_tokens = extract_candidate_phrases(clean_abstract, 'A')
final_body_tokens = extract_candidate_phrases(clean_body, 'B')

# Final output results
print("="*60)
print("FINAL STEP: SPACY PRECISION CLEANING TEST")
print("="*60)

print("TITLE FINAL TOKENS")
print(f"BEFORE (RE): {clean_title}")
print(f"AFTER (SPACY): {final_title_tokens}")
print("-" * 60)

print("ABSTRACT FINAL TOKENS (First 15)")
print(f"BEFORE (RE): {clean_abstract[:100]}...")
print(f"AFTER (SPACY): {final_abstract_tokens[:15]}")
print("-" * 60)

# Target check for noise removal
body_sample_re = clean_body[clean_body.find("include"):clean_body.find("figure")+6]
body_sample_spacy = extract_candidate_phrases(body_sample_re, 'B')

print("BODY FINAL TOKENS (Snippet Comparison)")
print(f"BEFORE (RE): {body_sample_re}")
print(f"AFTER (SPACY): {body_sample_spacy}")
print("="*60)

FINAL STEP: SPACY PRECISION CLEANING TEST
TITLE FINAL TOKENS
BEFORE (RE): detecting graph-based spatial outliers
AFTER (SPACY): ['detect', 'graph-base', 'spatial outlier']
------------------------------------------------------------
ABSTRACT FINAL TOKENS (First 15)
BEFORE (RE): of outliers can lead to the discovery of unexpected interesting and useful knowledge existing method...
AFTER (SPACY): ['outlier', 'lead', 'discovery', 'unexpected interesting', 'useful knowledge', 'exist', 'design', 'detect', 'spatial outlier', 'multidimensional geometric data set', 'distance metric', 'available', 'paper', 'focus', 'detect']
------------------------------------------------------------
BODY FINAL TOKENS (Snippet Comparison)
BEFORE (RE): includes about nine hundred stations each of which contains one to four loop detectors depending on the number of lanes sensors embedded in the freeways monitor the occupancy and volume of traffic on the road at regular intervals this information is sent to the t

# 5. Pipeline Evaluation & Quality Check

In [67]:
# Save final results to a file for verification
file_name = "final_preprocessing_check.txt"

with open(file_name, "w", encoding="utf-8") as f:
    f.write("=" * 60 + "\n")
    f.write("PROJECT: Structure-Aware Keyword Extraction\n")
    f.write(f"DOCUMENT ID: {test_sample['id']}\n")
    f.write("=" * 60 + "\n\n")

    f.write("[1. TITLE SECTION]\n")
    f.write(f"RAW: {raw_title}\n")
    f.write(f"CLEANED (RE): {clean_title}\n")
    f.write(f"FINAL (SPACY): {final_title_tokens}\n\n")

    f.write("-" * 60 + "\n")

    f.write("[2. ABSTRACT SECTION]\n")
    f.write(f"RAW (First 300): {raw_abstract[:300]}...\n")
    f.write(f"CLEANED (RE): {clean_abstract[:300]}...\n")
    f.write(f"FINAL (SPACY): {final_abstract_tokens}\n\n")

    f.write("-" * 60 + "\n")

    f.write("[3. BODY SECTION - NOISE CHECK]\n")
    target_phrase = "include i-"
    start_idx = raw_body.lower().find(target_phrase)
    if start_idx != -1:
        snippet_raw = raw_body[start_idx:start_idx+500]
        snippet_re = clean_academic_text(snippet_raw)
        snippet_spacy = extract_candidate_phrases(snippet_re, 'B')

        f.write(f"RAW SNIPPET: ...{snippet_raw}...\n\n")
        f.write(f"CLEANED (RE) SNIPPET: {snippet_re}\n\n")
        f.write(f"FINAL (SPACY) TOKENS: {snippet_spacy}\n")
    else:
        f.write("Target noise phrase not found in this specific snippet, but check full processing.\n")

    f.write("\n" + "="*60 + "\n")
    f.write("GROUND TRUTH (Reference): " + ", ".join(test_sample['extractive_keyphrases']) + "\n")
    f.write("="*60 + "\n")

print(f"File '{file_name}' created successfully. Download it to check final output.")

File 'final_preprocessing_check.txt' created successfully. Download it to check final output.


# Wrapping-up

In [68]:
import pickle

export_data = []
print("Building structured data... (Processing all documents)")

for sample in dataset['test']:
    structured = segment_and_structure(sample['document'])

    title_p = extract_candidate_phrases(clean_academic_text(structured['title']), section='T')
    abstract_p = extract_candidate_phrases(clean_academic_text(structured['abstract']), section='A')
    body_p = extract_candidate_phrases(clean_academic_text(structured['body']), section='B')

    export_data.append({
        "doc_id": sample['id'],
        "all_phrases": title_p + abstract_p + body_p,
        "title_phrases": title_p,
        "abstract_phrases": abstract_p,
        "body_phrases": body_p,
        "ground_truth": sample['extractive_keyphrases']
    })

file_name = "preprocessed_krapivin.pkl"
with open(file_name, "wb") as f:
    pickle.dump(export_data, f)

print(f"Final data generated: {file_name}")
print(f"Total processed documents: {len(export_data)}")

Building structured data for partner... (Processing all documents)
Final data generated: preprocessed_krapivin.pkl
Total processed documents: 2304


In [70]:
import pickle

# Reload the saved file
with open("preprocessed_krapivin.pkl", "rb") as f:
    loaded_data = pickle.load(f)

# Check the data structure of the first document (Index 0)
sample = loaded_data[0]

print("="*70)
print(f"DOCUMENT ID: {sample['doc_id']}")
print("="*70)

# 1. Check phrase counts and previews by section
print(f"TITLE PHRASES ({len(sample['title_phrases'])} items):")
print(f"   {sample['title_phrases']}")
print("-" * 70)

print(f"ABSTRACT PHRASES ({len(sample['abstract_phrases'])} items):")
print(f"   {sample['abstract_phrases'][:5]} ...") # Preview first 5 items
print("-" * 70)

print(f"BODY PHRASES (Confirming order preservation for first 10 of {len(sample['body_phrases'])} total items):")
# Preservation of order is important for calculating position weights.
for i, phrase in enumerate(sample['body_phrases'][:10]):
    print(f"   [{i}] {phrase}")
print("-" * 70)

# 2. Check total combined list (for TF-IDF)
print(f"ALL PHRASES (Total Count): {len(sample['all_phrases'])}")

# 3. Check Ground Truth
print("-" * 70)
print(f"GROUND TRUTH:")
print(f"   {sample['ground_truth']}")
print("="*70)

DOCUMENT ID: 502567
TITLE PHRASES (3 items):
   ['detect', 'graph-base', 'spatial outlier']
----------------------------------------------------------------------
ABSTRACT PHRASES (40 items):
   ['outlier', 'lead', 'discovery', 'unexpected interesting', 'useful knowledge'] ...
----------------------------------------------------------------------
BODY PHRASES (Confirming order preservation for first 10 of 1521 total items):
   [0] datum mining
   [1] process
   [2] nontrivial
   [3] unknown
   [4] useful
   [5] mation
   [6] knowledge rule
   [7] regularity
   [8] datum
   [9] database
----------------------------------------------------------------------
ALL PHRASES (Total Count): 1564
----------------------------------------------------------------------
GROUND TRUTH:
   ['outlier detection', 'spatial data mining']
